**The regression-based Leung's experiments**

In [1]:
def count_elements(arr):
    counts = {'1': 0, '-1': 0, '0': 0}
    for element in arr:
        if element == 1:
            counts['1'] += 1
        elif element == -1:
            counts['-1'] += 1
        elif element == 0:
            counts['0'] += 1
    return counts

arr = [1, -1, 0, 1, 1, -1, 0, 0, 1, -1]
element_counts = count_elements(arr)
print(element_counts)

{'1': 4, '-1': 3, '0': 3}


**CASE I (Baseline:) Unadjusted HT/Haj estimator**

In [2]:
import numpy as np, networkx as nx, math
from inference_module import *
from data_module import *

np.set_printoptions(threshold = np.inf) 


Y,X,D,A,A_norm,pscores0,IDs = assemble_data()

print('D:', D)
print('len(D):', len(D) )
print('D:', count_elements(D))

A = A.to_undirected()
bandwidths = range(4)
school_IDs = [24, 56, 22, 60, 58]

# summary statistics
# network_stats(A, IDs, school_IDs)

##### Treatment effect #####

print('Treatment effect')

# units eligible for treatment
pop = (D >= 0)
print('D:', D)




print('pop', pop)
print('len(pop)', len(pop))

# summary statistics
node_stats(Y[pop], D[pop])
treat_pop_size = Y[pop].size

print('Y:', Y)
print('len(Y):', len(Y))

print('X:', X)
print('len(X):', len(X))


# results
n = Y.size #3306个
Z = make_Zs(  Y,    D,          1-D,         0.5*np.ones(n),0.5*np.ones(n),pop)  #Here is the treated - untreated
Z1 = make_Zs( Y,    D,  np.zeros(n),         0.5*np.ones(n),0.5*np.ones(n),pop)  # here is only the treated
Z0 = -make_Zs(Y,    np.zeros(n),1-D,         0.5*np.ones(n),0.5*np.ones(n),pop) #here is only the untreated   #These data are all observed!
estimate_treat = np.array([Z[pop].mean(),Z1[pop].mean(),Z0[pop].mean()]) # compute the estimation
SE_treat = np.array([network_SE(Z, A, pop, 1, True, False, b) for b in bandwidths])  # compute the variance


# 按照原来的设计，pop总共320个
print('Z1:', len(Z1))
print('Z0:', len(Z0))




print('Z1[pop]:', len(Z1[pop]))
print('Z0[pop]:', len(Z0[pop]))


print('Z1[pop]-value', (Z1[pop]))
print('Z0[pop]-value', (Z0[pop]))


##here is the direct  treatment effect...


#回归调整的G = D (160*2=320维向量)，尝试两种方案：1）做回归调整； 
#1) 确定这些点的X值，利用gender+grade构成自身的协变量；grade大家都一样啊。。先不管,已经提取出来了
#确定\beta 回归： \beta = 待做！！！


##### Spillover effect #####

print('\n\nSpillover effect')

# restrict to units with a friend eligible for treatment
pop = (np.squeeze(np.asarray(A_norm.dot((D>=0)[:,None]))) > 0)
has_treated_friends = (np.squeeze(np.asarray(A_norm.dot((D==1)[:,None]))) > 0) # > 0 treated friends

# summary statistics
node_stats(Y[pop], has_treated_friends[pop])
spill_pop_size = Y[pop].size

# results

n = Y.size 
Z = make_Zs(Y,   has_treated_friends,1-has_treated_friends,  1-pscores0,pscores0,pop)
Z1 = make_Zs(Y,  has_treated_friends,np.zeros(n),            1-pscores0,pscores0,pop)
Z0 = -make_Zs(Y, np.zeros(n),1-has_treated_friends,          1-pscores0,pscores0,pop)

estimate_spill = np.array([Z[pop].mean(),Z1[pop].mean(),Z0[pop].mean()])
SE_spill = np.array([network_SE(Z, A, pop, 1, True, False, b) for b in bandwidths])

# 按照原来的设计，pop总共320个
print('Z1:', len(Z1))
print('Z0:', len(Z0))



print('Z1[pop]:', len(Z1[pop]))
print('Z0[pop]:', len(Z0[pop]))


print('Z1[pop]-value', (Z1[pop]))
print('Z0[pop]-value', (Z0[pop]))


# mu(0) results
SE_mu0 = np.array([network_SE(Z0, A, pop, 1, True, False, b) for b in bandwidths])



##### Print results #####

table = pd.DataFrame(np.vstack([np.hstack([estimate_treat, SE_treat]), np.hstack([estimate_spill, SE_spill])]).T)
table.index = ['$\hat\\tau(1,0)$', '$\hat\\mu(1)$', '$\hat\\mu(0)$'] + ['$b_n = ' + str(i) + '$' for i in bandwidths]
table.columns = ['Treatment', 'Spillover']
print('\n\n\\begin{table}[ht]')
print('\centering')
print('\caption{Causal Effect Estimates and Standard Errors (HT)}')
print('\\begin{threeparttable}')
print(table.to_latex(float_format = lambda x: '%.4f' % x, header=True, escape=False))
print('\\begin{tablenotes}[para,flushleft]')
print("\\footnotesize Columns display results for the treatment ($n={}$) and spillover ($n={}$) effects. Rows ``$b_n=k$'' report standard errors for the indicated values of the bandwidth.".format(treat_pop_size,spill_pop_size))
print('\end{tablenotes}')
print('\end{threeparttable}')
print('\end{table}')

print('\n\n\hat\mu(0) SEs')
print(SE_mu0)










data.columns: Index(['SCHID', 'UID', 'SCHTREAT', 'ID', 'TREAT', 'STRB', 'GENDER', 'ST1',
       'ST2', 'ST3', 'ST4', 'ST5', 'ST6', 'ST7', 'ST8', 'ST9', 'ST10',
       'WRISTOW2', 'GRADESWELLWORKWITH', 'GRADE_LEVEL'],
      dtype='object')
data_columns: Index(['SCHID', 'UID', 'SCHTREAT', 'ID', 'TREAT', 'STRB', 'GENDER', 'ST1',
       'ST2', 'ST3', 'ST4', 'ST5', 'ST6', 'ST7', 'ST8', 'ST9', 'ST10',
       'WRISTOW2', 'GRADESWELLWORKWITH', 'GRADE_LEVEL'],
      dtype='object')
D: [-1.  0.  0. -1. -1.  0. -1. -1. -1. -1. -1. -1. -1. -1. -1. -1. -1. -1.
  1. -1. -1. -1. -1. -1. -1. -1. -1.  0.  1. -1. -1. -1. -1. -1. -1. -1.
  0. -1. -1. -1. -1. -1. -1. -1. -1. -1. -1. -1. -1. -1. -1. -1. -1.  1.
 -1. -1. -1. -1. -1. -1. -1. -1.  1. -1.  0. -1. -1. -1. -1. -1. -1. -1.
 -1. -1. -1. -1. -1. -1. -1. -1. -1. -1. -1. -1. -1.  0. -1. -1. -1. -1.
 -1. -1. -1.  1. -1. -1. -1. -1. -1. -1.  1. -1. -1. -1. -1. -1. -1. -1.
 -1. -1. -1. -1. -1. -1. -1. -1. -1. -1. -1. -1. -1. -1. -1. -1. -1. -1.
  1. -1.


\begin{tablenotes}[para,flushleft]
\footnotesize $n= 320$.
\end{tablenotes}
\end{threeparttable}
\end{table}
Y: [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 1. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 1. 0.
 0. 0. 0. 0. 1. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0

Z1: 3302
Z0: 3302
Z1[pop]: 320
Z0[pop]: 320
Z1[pop]-value [0. 0. 0. 0. 0. 2. 0. 0. 0. 0. 0. 0. 2. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 2. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 2. 0. 0. 2. 0. 0. 0. 0. 2. 0. 0. 0. 0. 0. 0. 0.
 2. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 2. 0. 2. 0. 2. 0. 2. 2. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 2. 2. 0. 0. 2. 0. 0. 2. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 2. 2. 0. 0. 0. 0. 2. 0. 0. 2. 0. 0. 0. 0. 0. 0. 2. 2.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 2. 0. 0. 0. 0. 0. 0. 0. 2. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 2. 0. 0. 2. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 2. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 2. 0. 0. 0. 0. 0. 2. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 2. 2. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 2. 0. 2. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 2. 0. 0. 0. 0. 0. 0. 0. 0. 0. 2. 2. 2. 0. 0. 0. 0. 0. 2. 0.



\begin{table}[ht]
\centering
\caption{Causal Effect Estimates and Standard Errors (HT)}
\begin{threeparttable}
\begin{tabular}{lrr}
\toprule
{} &  Treatment &  Spillover \\
\midrule
$\hat\tau(1,0)$ &     0.1500 &     0.0408 \\
$\hat\mu(1)$    &     0.2375 &     0.1296 \\
$\hat\mu(0)$    &     0.0875 &     0.0887 \\
$b_n = 0$       &     0.0443 &     0.0168 \\
$b_n = 1$       &     0.0460 &     0.0184 \\
$b_n = 2$       &     0.0394 &     0.0205 \\
$b_n = 3$       &     0.0470 &     0.0171 \\
\bottomrule
\end{tabular}

\begin{tablenotes}[para,flushleft]
\footnotesize Columns display results for the treatment ($n=320$) and spillover ($n=1681$) effects. Rows ``$b_n=k$'' report standard errors for the indicated values of the bandwidth.
\end{tablenotes}
\end{threeparttable}
\end{table}


\hat\mu(0) SEs
[0.01191212 0.01225255 0.01503929 0.01672755]


In [3]:
import numpy as np, networkx as nx, math
from inference_module import *
from data_module import *

np.set_printoptions(threshold = np.inf) 


Y,X,D,A,A_norm,pscores0,IDs = assemble_data()

print('D:', D)
print('len(D):', len(D) )
print('D:', count_elements(D))

A = A.to_undirected()


bandwidths = range(4)
school_IDs = [24, 56, 22, 60, 58]

# summary statistics
# network_stats(A, IDs, school_IDs)

##### Treatment effect #####

print('Treatment effect')

# units eligible for treatment
pop = (D >= 0)
print('D:', D)




print('pop', pop)
print('len(pop)', len(pop))

# summary statistics
node_stats(Y[pop], D[pop])
treat_pop_size = Y[pop].size

print('Y:', Y)
print('len(Y):', len(Y))

print('X:', X)
print('len(X):', len(X))


# results
n = Y.size #3302个
Z = make_Zs_haj(  Y,    D,          1-D,         0.5*np.ones(n),0.5*np.ones(n),pop)  #Here is the treated - untreated
Z1 = make_Zs_haj( Y,    D,  np.zeros(n),         0.5*np.ones(n),0.5*np.ones(n),pop)  # here is only the treated
Z0 = -make_Zs_haj(Y,    np.zeros(n),1-D,         0.5*np.ones(n),0.5*np.ones(n),pop) #here is only the untreated   #These data are all observed!
estimate_treat = np.array([Z[pop].mean(),Z1[pop].mean(),Z0[pop].mean()]) # compute the estimation
SE_treat = np.array([network_SE(Z, A, pop, 1, True, False, b) for b in bandwidths])  # compute the variance


# 按照原来的设计，pop总共320个
print('Z1:', len(Z1))
print('Z0:', len(Z0))




print('Z1[pop]:', len(Z1[pop]))
print('Z0[pop]:', len(Z0[pop]))


print('Z1[pop]-value', (Z1[pop]))
print('Z0[pop]-value', (Z0[pop]))


##here is the direct  treatment effect...


#回归调整的G = D (160*2=320维向量)，尝试两种方案：1）做回归调整； 
#1) 确定这些点的X值，利用gender+grade构成自身的协变量；grade大家都一样啊。。先不管,已经提取出来了
#确定\beta 回归： \beta = 待做！！！


##### Spillover effect #####

print('\n\nSpillover effect')

# restrict to units with a friend eligible for treatment
pop = (np.squeeze(np.asarray(A_norm.dot((D>=0)[:,None]))) > 0)
has_treated_friends = (np.squeeze(np.asarray(A_norm.dot((D==1)[:,None]))) > 0) # > 0 treated friends

# summary statistics
node_stats(Y[pop], has_treated_friends[pop])
spill_pop_size = Y[pop].size

# results

n = Y.size 
Z = make_Zs_haj(Y,   has_treated_friends,1-has_treated_friends,  1-pscores0,pscores0,pop)
Z1 = make_Zs_haj(Y,  has_treated_friends,np.zeros(n),            1-pscores0,pscores0,pop)
Z0 = -make_Zs_haj(Y, np.zeros(n),1-has_treated_friends,          1-pscores0,pscores0,pop)





estimate_spill = np.array([Z[pop].mean(),Z1[pop].mean(),Z0[pop].mean()])
SE_spill = np.array([network_SE(Z, A, pop, 1, True, False, b) for b in bandwidths])

# 按照原来的设计，pop总共320个
print('Z1:', len(Z1))
print('Z0:', len(Z0))



print('Z1[pop]:', len(Z1[pop]))
print('Z0[pop]:', len(Z0[pop]))


print('Z1[pop]-value', (Z1[pop]))
print('Z0[pop]-value', (Z0[pop]))


# mu(0) results
SE_mu0 = np.array([network_SE(Z0, A, pop, 1, True, False, b) for b in bandwidths])



##### Print results #####

table = pd.DataFrame(np.vstack([np.hstack([estimate_treat, SE_treat]), np.hstack([estimate_spill, SE_spill])]).T)
table.index = ['$\hat\\tau(1,0)$', '$\hat\\mu(1)$', '$\hat\\mu(0)$'] + ['$b_n = ' + str(i) + '$' for i in bandwidths]
table.columns = ['Treatment', 'Spillover']
print('\n\n\\begin{table}[ht]')
print('\centering')
print('\caption{Causal Effect Estimates and Standard Errors (haj)}')
print('\\begin{threeparttable}')
print(table.to_latex(float_format = lambda x: '%.4f' % x, header=True, escape=False))
print('\\begin{tablenotes}[para,flushleft]')
print("\\footnotesize Columns display results for the treatment ($n={}$) and spillover ($n={}$) effects. Rows ``$b_n=k$'' report standard errors for the indicated values of the bandwidth.".format(treat_pop_size,spill_pop_size))
print('\end{tablenotes}')
print('\end{threeparttable}')
print('\end{table}')

print('\n\n\hat\mu(0) SEs')
print(SE_mu0)










data_columns: Index(['SCHID', 'UID', 'SCHTREAT', 'ID', 'TREAT', 'STRB', 'GENDER', 'ST1',
       'ST2', 'ST3', 'ST4', 'ST5', 'ST6', 'ST7', 'ST8', 'ST9', 'ST10',
       'WRISTOW2', 'GRADESWELLWORKWITH', 'GRADE_LEVEL'],
      dtype='object')
D: [-1.  0.  0. -1. -1.  0. -1. -1. -1. -1. -1. -1. -1. -1. -1. -1. -1. -1.
  1. -1. -1. -1. -1. -1. -1. -1. -1.  0.  1. -1. -1. -1. -1. -1. -1. -1.
  0. -1. -1. -1. -1. -1. -1. -1. -1. -1. -1. -1. -1. -1. -1. -1. -1.  1.
 -1. -1. -1. -1. -1. -1. -1. -1.  1. -1.  0. -1. -1. -1. -1. -1. -1. -1.
 -1. -1. -1. -1. -1. -1. -1. -1. -1. -1. -1. -1. -1.  0. -1. -1. -1. -1.
 -1. -1. -1.  1. -1. -1. -1. -1. -1. -1.  1. -1. -1. -1. -1. -1. -1. -1.
 -1. -1. -1. -1. -1. -1. -1. -1. -1. -1. -1. -1. -1. -1. -1. -1. -1. -1.
  1. -1. -1. -1. -1. -1. -1. -1. -1. -1. -1. -1. -1. -1. -1. -1. -1. -1.
 -1. -1. -1.  0. -1. -1. -1. -1. -1. -1. -1.  0. -1. -1. -1. -1. -1. -1.
 -1. -1. -1. -1. -1. -1. -1. -1. -1. -1. -1. -1. -1. -1. -1. -1. -1. -1.
 -1. -1. -1.  1. -1. -1. -1.

X: [1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 1. 1. 1. 1. 1. 1. 0. 1.
 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.
 1. 1. 1. 1. 1. 1. 0. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.
 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.
 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 0. 1. 0. 1. 1. 1. 1. 1.
 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.
 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.
 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.
 1. 1. 0. 1. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 1. 1.
 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 0. 0. 0. 0. 0. 0.
 0. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 0. 1. 1. 1. 0.

Z1: 3302
Z0: 3302
Z1[pop]: 320
Z0[pop]: 320
Z1[pop]-value [0. 0. 0. 0. 0. 2. 0. 0. 0. 0. 0. 0. 2. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 2. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 2. 0. 0. 2. 0. 0. 0. 0. 2. 0. 0. 0. 0. 0. 0. 0.
 2. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 2. 0. 2. 0. 2. 0. 2. 2. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 2. 2. 0. 0. 2. 0. 0. 2. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 2. 2. 0. 0. 0. 0. 2. 0. 0. 2. 0. 0. 0. 0. 0. 0. 2. 2.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 2. 0. 0. 0. 0. 0. 0. 0. 2. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 2. 0. 0. 2. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 2. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 2. 0. 0. 0. 0. 0. 2. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 2. 2. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 2. 0. 2. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 2. 0. 0. 0. 0. 0. 0. 0. 0. 0. 2. 2. 2. 0. 0. 0. 0. 0. 2. 0.

ps0: [0.5        0.5        0.23333333 0.5        0.5        0.5
 0.1        0.23333333 0.1        0.25       0.5        0.5
 0.23333333 0.23333333 0.23333333 0.5        0.23333333 0.5
 0.5        0.5        0.5        0.23333333 0.5        0.5
 0.5        0.5        0.5        0.25       0.5        0.5
 0.25       0.5        0.5        0.25       0.5        0.5
 0.25       0.5        0.23333333 0.5        0.25       0.5
 0.5        0.5        0.11666667 0.23333333 0.5        0.5
 0.5        0.25       0.5        0.23333333 0.5        0.23333333
 0.5        0.5        0.5        0.5        0.5        0.11666667
 0.5        0.5        0.5        0.5        0.5        0.5
 0.11666667 0.25       0.11666667 0.5        0.5        0.5
 0.5        0.11666667 0.5        0.5        0.5        0.5
 0.5        0.5        0.5        0.5        0.5        0.5
 0.5        0.25       0.5        0.5        0.5        0.23333333
 0.5        0.23333333 0.23333333 0.25       0.5        0.5
 0.5        0.

Z1: 3302
Z0: 3302
Z1[pop]: 1681
Z0[pop]: 1681
Z1[pop]-value [0.         0.         0.         0.         0.         0.
 0.         0.         0.         0.         0.         0.
 0.         0.         0.         0.         0.         0.
 0.         0.         0.         0.         0.         0.
 0.         0.         0.         0.         0.         0.
 0.         0.         0.         0.         0.         0.
 0.         0.         0.         0.         0.         0.
 0.         0.         0.         0.         0.         0.
 0.         0.         0.         0.         2.06453491 0.
 0.         0.         0.         0.         0.         0.
 0.         0.         0.         0.         0.         0.
 0.         0.         0.         0.         0.         0.
 0.         0.         0.         0.         0.         0.
 0.         0.         0.         0.         0.         0.
 0.         0.         0.         0.         0.         0.
 0.         0.         0.         0.         0.        



\begin{table}[ht]
\centering
\caption{Causal Effect Estimates and Standard Errors (haj)}
\begin{threeparttable}
\begin{tabular}{lrr}
\toprule
{} &  Treatment &  Spillover \\
\midrule
$\hat\tau(1,0)$ &     0.1500 &     0.0482 \\
$\hat\mu(1)$    &     0.2375 &     0.1337 \\
$\hat\mu(0)$    &     0.0875 &     0.0855 \\
$b_n = 0$       &     0.0443 &     0.0167 \\
$b_n = 1$       &     0.0460 &     0.0185 \\
$b_n = 2$       &     0.0394 &     0.0207 \\
$b_n = 3$       &     0.0470 &     0.0173 \\
\bottomrule
\end{tabular}

\begin{tablenotes}[para,flushleft]
\footnotesize Columns display results for the treatment ($n=320$) and spillover ($n=1681$) effects. Rows ``$b_n=k$'' report standard errors for the indicated values of the bandwidth.
\end{tablenotes}
\end{threeparttable}
\end{table}


\hat\mu(0) SEs
[0.01147993 0.01180801 0.01449364 0.01612065]


In [4]:
print('X:', X)
print('X.shape:', X.shape)  # 3302个数据点

print('Y:', Y)
print('Y.shape:', Y.shape)  # 3302个数据点


# print('pop.shape:', pop.shape)   
# assign = pop.astype(int)
# print('assign:', assign)
# print('assign.shape:', assign.shape)






G = A_norm   #这是归一化的邻接特征矩阵


E = A_norm
E.data[:] = 1  #construct the adjcent matrix for preparation. 这是0-1邻接矩阵



A_ = np.dot(E, E)   #这是A = E*E矩阵，符号与仿真实验相同























X: [1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 1. 1. 1. 1. 1. 1. 0. 1.
 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.
 1. 1. 1. 1. 1. 1. 0. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.
 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.
 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 0. 1. 0. 1. 1. 1. 1. 1.
 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.
 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.
 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.
 1. 1. 0. 1. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 1. 1.
 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 0. 0. 0. 0. 0. 0.
 0. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 0. 1. 1. 1. 0.

In [5]:
print('X:', X)

X: [1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 1. 1. 1. 1. 1. 1. 0. 1.
 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.
 1. 1. 1. 1. 1. 1. 0. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.
 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.
 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 0. 1. 0. 1. 1. 1. 1. 1.
 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.
 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.
 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.
 1. 1. 0. 1. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 1. 1.
 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 0. 0. 0. 0. 0. 0.
 0. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 0. 1. 1. 1. 0.

**Our regression (HT-PNI)**

In [6]:
from scipy.stats import binom, norm
from sklearn.preprocessing import scale
import numpy as np
from scipy.sparse.linalg import spsolve

print('Y1:', Y)

num_nb = np.sum(E, axis=1)

num_rep=10000

#direct treatment effect:

pop = (D >= 0)



def get_T(Z):
    return (E.dot(Z) > np.floor(num_nb/2)).astype(int)
#注意符号


def get_X(X, Z, G):
    return np.column_stack((Z, G.dot(Z), X, G.dot(X)))


# def get_orth_coef(X, G, num_rep=10000):
mom_mat = np.zeros((num_rep, 5))
for i in range(num_rep):
#         Z = np.random.binomial(1, r1, n)
    X_aug = get_X(X[pop], D[pop], A_norm[pop][:, pop])
#     T_vec = get_T(Z)
    w = D[pop]/(1-0.5) - (1 - D[pop])/0.5
    mom_mat[i] = [np.mean(w**2), np.mean(X_aug[:, 0]*w), np.mean(X_aug[:, 1]*w), np.mean(X_aug[:, 2]*w), np.mean(X_aug[:, 3]*w)]

mom_mean = np.mean(mom_mat, axis=0)
get_orth_coef = mom_mean[1:]/mom_mean[0]   #ouput: parametors.

print('get_orth_coef:', get_orth_coef)


## add the intercept term!
# def get_X_intercept(X, Z, G):
#     return np.column_stack(( np.ones(len(X)), Z, G.dot(Z), X, G.dot(X)))


# mom_mat = np.zeros((num_rep, 6))
# for i in range(num_rep):
# #         Z = np.random.binomial(1, r1, n)
#     X_aug = get_X_intercept(X[pop], assign[pop], A_norm[pop][:, pop])
# #     T_vec = get_T(Z)
#     w = assign[pop]/(1-0.5) - (1 - assign[pop])/0.5
#     mom_mat[i] = [np.mean(w**2), np.mean(X_aug[:, 0]*w), np.mean(X_aug[:, 1]*w), np.mean(X_aug[:, 2]*w), np.mean(X_aug[:, 3]*w), np.mean(X_aug[:, 4]*w)]

# mom_mean = np.mean(mom_mat, axis=0)
# get_orth_coef_intercept = mom_mean[1:]/mom_mean[0]

# print('get_orth_coef_intercept:', get_orth_coef_intercept)


X_aug = get_X(X[pop], D[pop], A_norm[pop][:, pop])
X_db_PNI = X_aug - np.dot(w.reshape(-1, 1), get_orth_coef.reshape(1, -1))
print('X_db_PNI', X_db_PNI)

print('Y2:', Y)


D_2 = scale(X_db_PNI * np.array(w).reshape(-1,1), axis=0, with_std=False)
A_ = np.dot(A_norm[pop][:, pop], A_norm[pop][:, pop])
matrix_left = (D_2.T @ A_) @ D_2
# matrix_left =np.array(matrix_left,dtype='float')
matrix_right = (D_2.T@ A_) @ (Y[pop]* w - Z[pop].mean())
# matrix_right =np.array(matrix_right,dtype='float')
hbeta_1 = spsolve(matrix_left, matrix_right)
print('Y3:', Y)
####有问题！！


hbeta_1 = [0.1,0.1,0.1,0.1]

Y[pop] = Y[pop] - np.dot(X_db_PNI[:,1:3], hbeta_1[1:3]) 

# Y_adj = Y

print('Y4:', Y)


#########################################################################regression-based experiment:
# Y_adj = Y
# Y_adj[pop] = Y[pop] - np.dot(X_db_PNI,    get_orth_coef)
#使用Y_new[pop]替换掉之前的Y,得到新的点估计

#test and debug:
# Y,X,D,A,A_norm,pscores0,IDs = assemble_data()
# Y_adj = Y




n = Y.size #3306个
Z = make_Zs(  Y,    D,          1-D,         0.5*np.ones(n),0.5*np.ones(n),pop)  #Here is the treated - untreated
Z1 = make_Zs( Y,    D,  np.zeros(n),         0.5*np.ones(n),0.5*np.ones(n),pop)  # here is only the treated
Z0 = -make_Zs(Y,    np.zeros(n),1-D,         0.5*np.ones(n),0.5*np.ones(n),pop) #here is only the untreated   #These data are all observed!
estimate_treat = np.array([Z[pop].mean(),Z1[pop].mean(),Z0[pop].mean()]) # compute the estimation
SE_treat = np.array([network_SE(Z, A, pop, 1, True, False, b) for b in bandwidths])  # compute the variance


# X_db = X_db_PNI


# D_2 = scale(X_db * np.array(w).reshape(-1,1), axis=0, with_std=False)
# #     print('D_2:', D_2)
# hbeta_1 = np.linalg.solve(np.dot(np.dot(D_2.T, A), D_2), np.dot(np.dot(D_2.T, A), (D - tau_unadj_ht)))
# hbeta_2 = np.linalg.solve(np.dot(np.dot(D_2.T, A_p), D_2), np.dot(np.dot(D_2.T, A_p), (D - tau_unadj_ht)))

# tau_adj_aug1 = np.mean((Y - np.dot(X_db, hbeta_1)) * w)
# tau_adj_aug2 = np.mean((Y - np.dot(X_db, hbeta_2)) * w)
    
# #     bias1 = np.mean((Y - np.dot(X_aug, hbeta_1)) * w) - tau_adj_aug1
# #     bias2 = np.mean((Y - np.dot(X_aug, hbeta_2)) * w) - tau_adj_aug2
# #     print('bias1:', bias1)
# #     print('bias2:', bias2)

# var_est_adj_aug1 = np.dot(np.dot((D - np.dot(D_2, hbeta_1) - tau_adj_aug1).T, A), (D - np.dot(D_2, hbeta_1) - tau_adj_aug1)) / n**2
# var_est_adj_aug2 = np.dot(np.dot((D - np.dot(D_2, hbeta_2) - tau_adj_aug2).T, A_p), (D - np.dot(D_2, hbeta_2) - tau_adj_aug2)) / n**2

# is_cover_adj_aug_1 = abs(tau_adj_aug1 - tau) < norm.ppf(0.975) * np.sqrt(var_est_adj_aug1)
# is_cover_adj_aug_2 = abs(tau_adj_aug2 - tau) < norm.ppf(0.975) * np.sqrt(var_est_adj_aug2)





# var_est_adj_aug1 = ((Y[pop]*w - np.dot(D_2, hbeta_1) - 0.15).T@ A_ @ (Y[pop]*w - np.dot(D_2, hbeta_1) - 0.15)) / len(D[pop])**2
# var_est_adj_aug1 = ((Y[pop]*w - np.dot(D_2, hbeta_1) - 0.15).T@ A_ @ (Y[pop]*w - np.dot(D_2, hbeta_1) - 0.15)) / len(D[pop])**2
print(np.sqrt(((Y[pop]*w - np.dot(X_db_PNI[:,1:3], hbeta_1[1:3])* w ).T@ A_ @ (Y[pop]*w - np.dot(X_db_PNI, hbeta_1)* w ))/ (len(Y[pop])**2)))



# print('var_est_adj_aug1:', var_est_adj_aug1)












# orth_coef_intercept = get_orth_coef_intercept(X,A_norm)


































Y1: [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 1. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 1. 0.
 0. 0. 0. 0. 1. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0

get_orth_coef: [ 0.25       0.0109375 -0.003125  -0.0078125]
X_db_PNI [[ 0.5       0.021875 -0.00625  -0.015625]
 [ 0.5       0.021875 -0.00625  -0.015625]
 [ 0.5       1.021875 -0.00625  -0.015625]
 [ 0.5      -0.021875  1.00625   0.015625]
 [ 0.5       0.021875  0.99375  -0.015625]
 [ 0.5      -0.021875  1.00625   0.015625]
 [ 0.5       0.021875  0.99375  -0.015625]
 [ 0.5      -0.021875  1.00625   0.015625]
 [ 0.5      -0.021875  0.00625   0.015625]
 [ 0.5       0.021875 -0.00625  -0.015625]
 [ 0.5       2.021875  0.99375   2.984375]
 [ 0.5       0.978125  1.00625   1.015625]
 [ 0.5      -0.021875  1.00625   0.015625]
 [ 0.5      -0.021875  1.00625   1.015625]
 [ 0.5       0.021875  0.99375  -0.015625]
 [ 0.5       1.021875  0.99375   2.984375]
 [ 0.5       0.978125  1.00625   1.015625]
 [ 0.5      -0.021875  1.00625   0.015625]
 [ 0.5       0.021875  0.99375  -0.015625]
 [ 0.5       1.021875  0.99375   1.984375]
 [ 0.5       0.021875  0.99375  -0.015625]
 [ 0.5      -0.021875  1.00

/Users/zhangzhiheng/opt/anaconda3/lib/python3.8/site-packages/scipy/sparse/linalg/dsolve/linsolve.py:144: SparseEfficiencyWarning: spsolve requires A be CSC or CSR matrix format
  warn('spsolve requires A be CSC or CSR matrix format',


0.02634055367805325


In [7]:
print(X_db_PNI)
print(w.shape)
print('w:', w)
print(np.dot(w.T, X_db_PNI))
print('hbeta1:', hbeta_1)
print('Y[pop]*w:', np.dot(Y[pop],w)/len(Y[pop]))
print('Y[pop]:', Y[pop])
print('np.dot(X_db_PNI[:,1:3], hbeta_1[1:3])* w:', np.dot(X_db_PNI, hbeta_1)* w)
print('X_db_PNI:', X_db_PNI)
print('hbeta_1:', hbeta_1)
print(np.sqrt(((Y[pop]*w - 0.5*np.dot(X_db_PNI, hbeta_1)* w ).T@ A_ @ (Y[pop]*w - 0.5*np.dot(X_db_PNI, hbeta_1)* w ))/ (len(Y[pop])**2)))


print((((Y[pop]*w - 0.15 ).T@ A_ @ (Y[pop]*w - 0.15))/ (len(Y[pop])**2)))

[[ 0.5       0.021875 -0.00625  -0.015625]
 [ 0.5       0.021875 -0.00625  -0.015625]
 [ 0.5       1.021875 -0.00625  -0.015625]
 [ 0.5      -0.021875  1.00625   0.015625]
 [ 0.5       0.021875  0.99375  -0.015625]
 [ 0.5      -0.021875  1.00625   0.015625]
 [ 0.5       0.021875  0.99375  -0.015625]
 [ 0.5      -0.021875  1.00625   0.015625]
 [ 0.5      -0.021875  0.00625   0.015625]
 [ 0.5       0.021875 -0.00625  -0.015625]
 [ 0.5       2.021875  0.99375   2.984375]
 [ 0.5       0.978125  1.00625   1.015625]
 [ 0.5      -0.021875  1.00625   0.015625]
 [ 0.5      -0.021875  1.00625   1.015625]
 [ 0.5       0.021875  0.99375  -0.015625]
 [ 0.5       1.021875  0.99375   2.984375]
 [ 0.5       0.978125  1.00625   1.015625]
 [ 0.5      -0.021875  1.00625   0.015625]
 [ 0.5       0.021875  0.99375  -0.015625]
 [ 0.5       1.021875  0.99375   1.984375]
 [ 0.5       0.021875  0.99375  -0.015625]
 [ 0.5      -0.021875  1.00625   2.015625]
 [ 0.5      -0.021875  1.00625   0.015625]
 [ 0.5     

In [8]:

##### Spillover effect #####

print('\n\nSpillover effect')
Y,X,D,A,A_norm,pscores0,IDs = assemble_data()

A = A.to_undirected()


# restrict to units with a friend eligible for treatment
pop = (np.squeeze(np.asarray(A_norm.dot((D>=0)[:,None]))) > 0)
has_treated_friends = (np.squeeze(np.asarray(A_norm.dot((D==1)[:,None]))) > 0) # > 0 treated friends

has_over_half_friends =  (np.squeeze(np.asarray(A_norm.dot(np.ones(A_norm.shape[0])))) > 8)
# print( A_norm.dot(np.ones(A_norm.shape[0])))



# def get_orth_coef(X, G, num_rep=10000):
mom_mat = np.zeros((num_rep, 5))

# print('assign:', assign)
# print('T_vec:', get_T(assign)[pop])

print('pscores0:', pscores0)


for i in range(num_rep):
#         Z = np.random.binomial(1, r1, n)
    X_aug = get_X(X[pop], D[pop], A_norm[pop][:, pop])
#     T_vec = get_T(assign)
    w = has_treated_friends[pop]/(1-pscores0)[pop] - (1 - has_treated_friends[pop])/np.array(pscores0)[pop]
    mom_mat[i] = [np.mean(w**2), np.mean(X_aug[:, 0]*w), np.mean(X_aug[:, 1]*w), np.mean(X_aug[:, 2]*w), np.mean(X_aug[:, 3]*w)]

mom_mean = np.mean(mom_mat, axis=0)
get_orth_coef = mom_mean[1:]/mom_mean[0]   #ouput: parametors.

print('get_orth_coef:', get_orth_coef)




X_aug = get_X(X[pop], D[pop], A_norm[pop][:, pop])
X_db_PNI = X_aug - np.dot(w.reshape(-1, 1), get_orth_coef.reshape(1, -1))
print('X_db_PNI', X_db_PNI)


# Y_adj = Y
D_2 = scale(X_db_PNI * np.array(w).reshape(-1,1), axis=0, with_std=False)
A_ = np.dot(A_norm[pop][:, pop], A_norm[pop][:, pop])
matrix_left = (D_2.T @ A_) @ D_2
# matrix_left =np.array(matrix_left,dtype='float')
matrix_right = (D_2.T@ A_) @ (Y[pop]* w - Z[pop].mean())
# matrix_right =np.array(matrix_right,dtype='float')
hbeta_1 = spsolve(matrix_left, matrix_right)

hbeta_1 = [0.1,0.1,0.1,0.1]

Y[pop] = Y[pop] - np.dot(X_db_PNI[:,1:3], hbeta_1[1:3]) 



# D_2 = scale(X_db_PNI * np.array(w).reshape(-1,1), axis=0, with_std=False)
# A_ = np.dot(A_norm[pop][:, pop], A_norm[pop][:, pop])
# hbeta_1 = np.linalg.solve(np.dot(np.dot(D_2.T, A), D_2), np.dot(np.dot(D_2.T, A_), (Y[pop]* w - Z[pop].mean())) )
# Y_adj[pop] = Y[pop] - np.dot(X_db_PNI, hbeta_1) 

    
    
#########################################################################regression-based experiment:
# Y_adj = Y
# Y_adj[pop] = Y[pop] - np.dot(X_db_PNI,    get_orth_coef)









# summary statistics
node_stats(Y[pop], has_treated_friends[pop])
spill_pop_size = Y[pop].size

# results

n = Y.size 
Z = make_Zs(Y,   has_treated_friends,1-has_treated_friends,  1-pscores0,pscores0,pop)
Z1 = make_Zs(Y,  has_treated_friends,np.zeros(n),            1-pscores0,pscores0,pop)
Z0 = -make_Zs(Y, np.zeros(n),1-has_treated_friends,          1-pscores0,pscores0,pop)





estimate_spill = np.array([Z[pop].mean(),Z1[pop].mean(),Z0[pop].mean()])
SE_spill = np.array([network_SE(Z, A, pop, 1, True, False, b) for b in bandwidths])


print('Z1:', len(Z1))
print('Z0:', len(Z0))



print('Z1[pop]:', len(Z1[pop]))
print('Z0[pop]:', len(Z0[pop]))


print('Z1[pop]-value', (Z1[pop]))
print('Z0[pop]-value', (Z0[pop]))


# mu(0) results
SE_mu0 = np.array([network_SE(Z0, A, pop, 1, True, False, b) for b in bandwidths])



##### Print results #####

table = pd.DataFrame(np.vstack([np.hstack([estimate_treat, SE_treat]), np.hstack([estimate_spill, SE_spill])]).T)
table.index = ['$\hat\\tau(1,0)$', '$\hat\\mu(1)$', '$\hat\\mu(0)$'] + ['$b_n = ' + str(i) + '$' for i in bandwidths]
table.columns = ['Treatment', 'Spillover']
print('\n\n\\begin{table}[ht]')
print('\centering')
print('\caption{Causal Effect Estimates and Standard Errors (HT)}')
print('\\begin{threeparttable}')
print(table.to_latex(float_format = lambda x: '%.4f' % x, header=True, escape=False))
print('\\begin{tablenotes}[para,flushleft]')
print("\\footnotesize Columns display results for the treatment ($n={}$) and spillover ($n={}$) effects. Rows ``$b_n=k$'' report standard errors for the indicated values of the bandwidth.".format(treat_pop_size,spill_pop_size))
print('\end{tablenotes}')
print('\end{threeparttable}')
print('\end{table}')

print('\n\n\hat\mu(0) SEs')
print(SE_mu0)



var_est_adj_aug1 = ((Y[pop]*w - np.dot(D_2, hbeta_1) - 0.04).T@ A_ @ (Y[pop]*w - np.dot(D_2, hbeta_1) - 0.04)) / len(D[pop])**2






Spillover effect
data_columns: Index(['SCHID', 'UID', 'SCHTREAT', 'ID', 'TREAT', 'STRB', 'GENDER', 'ST1',
       'ST2', 'ST3', 'ST4', 'ST5', 'ST6', 'ST7', 'ST8', 'ST9', 'ST10',
       'WRISTOW2', 'GRADESWELLWORKWITH', 'GRADE_LEVEL'],
      dtype='object')
pscores0: [0.5        1.         0.5        1.         1.         0.23333333
 0.5        1.         1.         1.         1.         1.
 0.5        1.         1.         0.5        0.1        0.23333333
 1.         0.1        0.25       1.         0.5        0.5
 1.         0.23333333 0.23333333 1.         1.         1.
 1.         1.         1.         0.23333333 1.         0.5
 1.         0.23333333 1.         0.5        0.5        0.5
 0.5        0.23333333 1.         0.5        1.         1.
 0.5        0.5        0.5        0.5        1.         1.
 1.         0.25       0.5        1.         0.5        1.
 0.25       1.         1.         0.5        0.5        0.25
 0.5        1.         0.5        1.         1.         0.25
 

get_orth_coef: [ 0.0200439   0.02268483 -0.00414268 -0.00100492]
X_db_PNI [[-9.59912197e-01 -3.54630338e-01  9.91714644e-01  3.97990155e-01]
 [ 4.00878028e-02 -5.26058909e-01 -8.28535604e-03 -2.00984514e-03]
 [-2.61442192e-02 -5.29588910e-01  5.40349307e-03  2.01310769e-01]
 [-1.04008780e+00 -5.45369662e-01  8.28535604e-03  2.00984514e-03]
 [-1.04008780e+00 -7.12036329e-01  8.28535604e-03  2.00984514e-03]
 [-1.04008780e+00 -4.53696620e-02  8.28535604e-03  2.00984514e-03]
 [-1.02227100e+00 -4.69649812e-01  1.00460298e+00  6.67783247e-01]
 [-1.02614422e+00 -6.29588910e-01  1.00540349e+00  7.01310769e-01]
 [-1.02227100e+00 -6.91872034e-01  1.00460298e+00  8.90005470e-01]
 [-1.02672520e+00 -1.30246441e-01  1.00552357e+00  3.01339897e-01]
 [-9.59912197e-01 -2.04630338e-01 -8.28535604e-03 -2.00984514e-03]
 [-1.04008780e+00 -7.12036329e-01  1.00828536e+00  3.35343178e-01]
 [-1.02614422e+00 -2.79588910e-01  1.00540349e+00  7.51310769e-01]
 [-1.02614422e+00 -5.29588910e-01  1.00540349e+00  7.01

/Users/zhangzhiheng/opt/anaconda3/lib/python3.8/site-packages/scipy/sparse/linalg/dsolve/linsolve.py:144: SparseEfficiencyWarning: spsolve requires A be CSC or CSR matrix format
  warn('spsolve requires A be CSC or CSR matrix format',


Z1: 3302
Z0: 3302
Z1[pop]: 1681
Z0[pop]: 1681
Z1[pop]-value [-0.00000000e+00  0.00000000e+00  6.83720109e-02  1.07416861e-01
  1.40750195e-01  7.41686119e-03 -5.94392404e-02 -4.90192934e-02
 -3.47478823e-02 -1.16703617e-01  0.00000000e+00 -5.92498055e-02
 -9.46714674e-02 -6.20627717e-02 -7.51062500e-02 -0.00000000e+00
  0.00000000e+00  4.74168612e-02  0.00000000e+00  0.00000000e+00
  1.47416861e-01  1.76473732e-02  0.00000000e+00  0.00000000e+00
  1.32416861e-01  1.62972417e-01  1.27416861e-01  9.66297161e-02
  1.27416861e-01  1.18527972e-01  0.00000000e+00  1.67416861e-01
  0.00000000e+00  0.00000000e+00  8.24168612e-02  8.74168612e-02
  2.99630494e-02  0.00000000e+00  9.01111413e-02  0.00000000e+00
 -8.55925061e-02 -0.00000000e+00 -1.52583139e-01 -3.25831388e-02
 -5.42274208e-02 -0.00000000e+00 -1.25916472e-01 -0.00000000e+00
 -3.25831388e-02 -1.07814728e-01 -1.52583139e-01  0.00000000e+00
  1.93241686e+00 -0.00000000e+00 -0.00000000e+00 -1.92583139e-01
 -0.00000000e+00 -0.00000000e+



\begin{table}[ht]
\centering
\caption{Causal Effect Estimates and Standard Errors (HT)}
\begin{threeparttable}
\begin{tabular}{lrr}
\toprule
{} &  Treatment &  Spillover \\
\midrule
$\hat\tau(1,0)$ &     0.1500 &     0.0408 \\
$\hat\mu(1)$    &     0.1553 &     0.1274 \\
$\hat\mu(0)$    &     0.0053 &     0.0866 \\
$b_n = 0$       &     0.0423 &     0.0170 \\
$b_n = 1$       &     0.0432 &     0.0189 \\
$b_n = 2$       &     0.0371 &     0.0205 \\
$b_n = 3$       &     0.0466 &     0.0177 \\
\bottomrule
\end{tabular}

\begin{tablenotes}[para,flushleft]
\footnotesize Columns display results for the treatment ($n=320$) and spillover ($n=1681$) effects. Rows ``$b_n=k$'' report standard errors for the indicated values of the bandwidth.
\end{tablenotes}
\end{threeparttable}
\end{table}


\hat\mu(0) SEs
[0.01196112 0.01271062 0.0163475  0.01845663]


In [9]:

print('X_db_PNI:', X_db_PNI)

print('w:', w)
print(np.dot(w.T, X_db_PNI))

print(np.dot(w.T, Y[pop])/len(Y[pop]))
print('has_over_half_friends:', has_over_half_friends)

print('has_treated_friends:', has_treated_friends)



X_db_PNI: [[-9.59912197e-01 -3.54630338e-01  9.91714644e-01  3.97990155e-01]
 [ 4.00878028e-02 -5.26058909e-01 -8.28535604e-03 -2.00984514e-03]
 [-2.61442192e-02 -5.29588910e-01  5.40349307e-03  2.01310769e-01]
 [-1.04008780e+00 -5.45369662e-01  8.28535604e-03  2.00984514e-03]
 [-1.04008780e+00 -7.12036329e-01  8.28535604e-03  2.00984514e-03]
 [-1.04008780e+00 -4.53696620e-02  8.28535604e-03  2.00984514e-03]
 [-1.02227100e+00 -4.69649812e-01  1.00460298e+00  6.67783247e-01]
 [-1.02614422e+00 -6.29588910e-01  1.00540349e+00  7.01310769e-01]
 [-1.02227100e+00 -6.91872034e-01  1.00460298e+00  8.90005470e-01]
 [-1.02672520e+00 -1.30246441e-01  1.00552357e+00  3.01339897e-01]
 [-9.59912197e-01 -2.04630338e-01 -8.28535604e-03 -2.00984514e-03]
 [-1.04008780e+00 -7.12036329e-01  1.00828536e+00  3.35343178e-01]
 [-1.02614422e+00 -2.79588910e-01  1.00540349e+00  7.51310769e-01]
 [-1.02614422e+00 -5.29588910e-01  1.00540349e+00  7.01310769e-01]
 [-1.02614422e+00 -4.29588910e-01  1.00540349e+00  4

In [10]:
print(A_norm)
print(np.squeeze(np.asarray(A_norm.dot((D==1)[:,None]))))
print( A_norm.dot(np.ones(A_norm.shape[0])))
print( A_norm.dot(D==1))

  (0, 20)	0.1
  (0, 35)	0.1
  (0, 36)	0.1
  (0, 106)	0.1
  (0, 107)	0.1
  (0, 114)	0.1
  (0, 125)	0.1
  (0, 132)	0.1
  (0, 200)	0.1
  (0, 314)	0.1
  (1, 45)	0.25
  (1, 49)	0.25
  (1, 275)	0.25
  (1, 287)	0.25
  (2, 7)	0.14285714285714285
  (2, 64)	0.14285714285714285
  (2, 102)	0.14285714285714285
  (2, 162)	0.14285714285714285
  (2, 252)	0.14285714285714285
  (2, 266)	0.14285714285714285
  (2, 309)	0.14285714285714285
  (3, 38)	0.16666666666666666
  (3, 39)	0.16666666666666666
  (3, 41)	0.16666666666666666
  (3, 67)	0.16666666666666666
  :	:
  (3299, 2983)	0.16666666666666666
  (3299, 3164)	0.16666666666666666
  (3299, 3170)	0.16666666666666666
  (3299, 3230)	0.16666666666666666
  (3299, 3253)	0.16666666666666666
  (3300, 2696)	0.1
  (3300, 3077)	0.1
  (3300, 3154)	0.1
  (3300, 3183)	0.1
  (3300, 3188)	0.1
  (3300, 3190)	0.1
  (3300, 3215)	0.1
  (3300, 3284)	0.1
  (3300, 3291)	0.1
  (3300, 3293)	0.1
  (3301, 2774)	0.1
  (3301, 2776)	0.1
  (3301, 2833)	0.1
  (3301, 2879)	0.1
  (3301, 2

[1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.
 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.
 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.
 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.
 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.
 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.
 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.
 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.
 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.
 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.
 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.
 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.
 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 0. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.
 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.

In [11]:
Y,X,D,A,A_norm,pscores0,IDs = assemble_data()
print(X_db_PNI)
print(w.shape)
print('w:', w)
print(np.dot(w.T, X_db_PNI))
print('hbeta1:', hbeta_1)
pop = (np.squeeze(np.asarray(A_norm.dot((D>=0)[:,None]))) > 0)
print('pop:', pop)




print('Y[pop]*w:', np.dot(Y[pop],w)/len(Y[pop]))
print('Y[pop]:', Y[pop])

print('var_est_adj_aug1:', var_est_adj_aug1)


data_columns: Index(['SCHID', 'UID', 'SCHTREAT', 'ID', 'TREAT', 'STRB', 'GENDER', 'ST1',
       'ST2', 'ST3', 'ST4', 'ST5', 'ST6', 'ST7', 'ST8', 'ST9', 'ST10',
       'WRISTOW2', 'GRADESWELLWORKWITH', 'GRADE_LEVEL'],
      dtype='object')
[[-9.59912197e-01 -3.54630338e-01  9.91714644e-01  3.97990155e-01]
 [ 4.00878028e-02 -5.26058909e-01 -8.28535604e-03 -2.00984514e-03]
 [-2.61442192e-02 -5.29588910e-01  5.40349307e-03  2.01310769e-01]
 [-1.04008780e+00 -5.45369662e-01  8.28535604e-03  2.00984514e-03]
 [-1.04008780e+00 -7.12036329e-01  8.28535604e-03  2.00984514e-03]
 [-1.04008780e+00 -4.53696620e-02  8.28535604e-03  2.00984514e-03]
 [-1.02227100e+00 -4.69649812e-01  1.00460298e+00  6.67783247e-01]
 [-1.02614422e+00 -6.29588910e-01  1.00540349e+00  7.01310769e-01]
 [-1.02227100e+00 -6.91872034e-01  1.00460298e+00  8.90005470e-01]
 [-1.02672520e+00 -1.30246441e-01  1.00552357e+00  3.01339897e-01]
 [-9.59912197e-01 -2.04630338e-01 -8.28535604e-03 -2.00984514e-03]
 [-1.04008780e+00 -7.120

In [12]:
print(len(D[pop]))

D_2 = scale(X_db_PNI * np.array(w).reshape(-1,1), axis=0, with_std=False)


print(np.sqrt(((Y[pop]*w - np.dot(X_db_PNI, hbeta_1)* w ).T@ A_ @ (Y[pop]*w - np.dot(X_db_PNI, hbeta_1)* w ))/ (len(Y[pop])**2)))


print(np.sqrt(((Y[pop]*w - 0.04 ).T@ A_ @ (Y[pop]*w - 0.04 ))/ (len(Y[pop])**2)))

print('X_db_PNI:', X_db_PNI)
hbeta_1 = [1,1,1,1]
print('hbeta_1:', hbeta_1)
print('np.dot(X_db_PNI, hbeta_1)* w :', np.dot(X_db_PNI, hbeta_1)* w )
print('Y[pop]*w:', Y[pop]*w)

print('D_2:', D_2)
print('Y[pop]*w:', Y[pop]*w)
print('np.dot(D_2, hbeta_1):', np.dot(D_2, hbeta_1))
print('Y:', Y)

1681
0.0041928238237443505
0.0034778366995172416
X_db_PNI: [[-9.59912197e-01 -3.54630338e-01  9.91714644e-01  3.97990155e-01]
 [ 4.00878028e-02 -5.26058909e-01 -8.28535604e-03 -2.00984514e-03]
 [-2.61442192e-02 -5.29588910e-01  5.40349307e-03  2.01310769e-01]
 [-1.04008780e+00 -5.45369662e-01  8.28535604e-03  2.00984514e-03]
 [-1.04008780e+00 -7.12036329e-01  8.28535604e-03  2.00984514e-03]
 [-1.04008780e+00 -4.53696620e-02  8.28535604e-03  2.00984514e-03]
 [-1.02227100e+00 -4.69649812e-01  1.00460298e+00  6.67783247e-01]
 [-1.02614422e+00 -6.29588910e-01  1.00540349e+00  7.01310769e-01]
 [-1.02227100e+00 -6.91872034e-01  1.00460298e+00  8.90005470e-01]
 [-1.02672520e+00 -1.30246441e-01  1.00552357e+00  3.01339897e-01]
 [-9.59912197e-01 -2.04630338e-01 -8.28535604e-03 -2.00984514e-03]
 [-1.04008780e+00 -7.12036329e-01  1.00828536e+00  3.35343178e-01]
 [-1.02614422e+00 -2.79588910e-01  1.00540349e+00  7.51310769e-01]
 [-1.02614422e+00 -5.29588910e-01  1.00540349e+00  7.01310769e-01]
 [-

D_2: [[ 1.91982439e+00  7.09260676e-01 -1.98342929e+00 -7.95980310e-01]
 [-8.01756055e-02  1.05211782e+00  1.65707121e-02  4.01969027e-03]
 [-3.41011555e-02 -6.90768143e-01  7.04803444e-03  2.62579263e-01]
 [-2.08017561e+00 -1.09073932e+00  1.65707121e-02  4.01969027e-03]
 [-2.08017561e+00 -1.42407266e+00  1.65707121e-02  4.01969027e-03]
 [-2.08017561e+00 -9.07393240e-02  1.65707121e-02  4.01969027e-03]
 [-1.13585667e+00 -5.21833125e-01  1.11622553e+00  7.41981386e-01]
 [-1.33844898e+00 -8.21202926e-01  1.31139586e+00  9.14753176e-01]
 [-1.13585667e+00 -7.68746705e-01  1.11622553e+00  9.88894966e-01]
 [-1.36896694e+00 -1.73661922e-01  1.34069809e+00  4.01786529e-01]
 [ 1.91982439e+00  4.09260676e-01  1.65707121e-02  4.01969027e-03]
 [-2.08017561e+00 -1.42407266e+00  2.01657071e+00  6.70686357e-01]
 [-1.33844898e+00 -3.64681187e-01  1.31139586e+00  9.79970568e-01]
 [-1.33844898e+00 -6.90768143e-01  1.31139586e+00  9.14753176e-01]
 [-1.33844898e+00 -5.60333361e-01  1.31139586e+00  5.2344

 [-1.36896694e+00 -7.06995255e-01  1.34069809e+00  6.68453196e-01]]
Y[pop]*w: [-0.         -0.          0.          0.          0.          0.
  0.          0.          0.          0.         -0.          0.
  0.          0.          0.         -0.         -0.          0.
 -0.         -0.          0.          0.         -0.         -0.
  0.          0.          0.          0.          0.          0.
 -0.          0.         -0.         -0.          0.          0.
  0.         -0.          0.         -0.          0.         -0.
  0.          0.          0.         -0.          0.         -0.
  0.          0.          0.         -0.          2.         -0.
 -0.          0.         -0.         -0.          0.          0.
  0.          0.          0.         -0.          0.          0.
  0.          0.          0.          0.         -0.         -0.
 -0.          0.          0.          0.         -0.          0.
 -0.         -0.         -0.         -0.         -0.         -0.
  0.        

In [13]:
print(np.dot(X_db_PNI, hbeta_1).shape)


(1681,)
